In [1]:
!pip install quimb mps-to-circuit qiskit qiskit_aer

In [2]:
"""
main.py -- first-order Trotter sweep of the ground-state filter on a grid of n
equal steps (T = sum t_i = n * dt, every pulse t_i = k_i * dt).

Noiseless by design: exact statevector / exact-unitary numpy propagation
throughout, so this isolates algorithmic (Trotter + filter-design + DMRG)
error rather than device error. See builder.py's module docstring.

Requires the cleaned builder.py (explicit-argument helpers) and your
get_energies / mps_to_circuit modules.
"""
import csv

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from qiskit.quantum_info import SparsePauliOp

from builder import (
    FilterBuilder, MPO_ham_j1j2, get_energies, get_spectrum, j1j2_hamiltonian,
    alpha_comm, apply_filter_pulse, build_rodeo_circuit, cx_depth_per_step,
    grid_filter, postselected_run, postselected_run_exact, resource_costs,
    state_metrics, verify_pulse_convention,
)
from mps_to_circuit import mps_to_circuit

# ----------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------
N = 8
J1, J2 = 1.0, 0.7
N_SWEEP = [10, 20, 40, 80, 160, 320]   # total Trotter steps n (dt = T / n)
SWEEP_L = 1                            # trial-circuit layers
VERIFY_COST_NS = (N_SWEEP[0], N_SWEEP[-1])   # full transpile at both ends of
                                              # N_SWEEP, to check whether the
                                              # linear cx1*n estimate holds
                                              # equally well at small and large n
ORDER = 1                              # first-order Trotter

# ----------------------------------------------------------------------
# 0. Convention check -- everything below depends on this being right
# ----------------------------------------------------------------------
if not verify_pulse_convention():
    raise RuntimeError(
        "verify_pulse_convention() failed: the circuit's success-branch / "
        "phase-sign convention doesn't match FilterBuilder.apply_filter. "
        "Fix this before trusting any fidelity or P_succ number below.")

# ----------------------------------------------------------------------
# Hamiltonian, reference ground states, spectrum
# ----------------------------------------------------------------------
H_mpo = MPO_ham_j1j2(N, j1=J1, j2=J2, cyclic=False)
res = get_energies(H_mpo, N=N, j1=J1, j2=J2, run_ed_check=True)
E0_dmrg, psi0_mps, ed = float(np.real(res["E0"])), res["psi0"], res["ed"]
if ed is None:
    raise RuntimeError("ED cross-check didn't run (check ED_CHECK_MAX_N / N).")
psi0_ed = ed["psi0_ed"]

psi0_dmrg = np.asarray(psi0_mps.to_dense()).reshape(-1)
psi0_dmrg = psi0_dmrg / np.linalg.norm(psi0_dmrg)

# F_dmrg (the headline metric everywhere below) is only as good as this
# overlap -- DMRG is itself an approximate ground state.
dmrg_ed_overlap = float(abs(np.vdot(psi0_ed, psi0_dmrg)) ** 2)
print(f"E0 DMRG = {E0_dmrg:.8f}   |<DMRG|ED>|^2 = {dmrg_ed_overlap:.8f}")
if dmrg_ed_overlap < 1 - 1e-4:
    print(f"  [note] DMRG-ED overlap is {dmrg_ed_overlap:.6f}, not 1: "
          f"F_dmrg numbers below inherit up to ~{1 - dmrg_ed_overlap:.1e} "
          f"of DMRG's own error, not just Trotter/filter error.")

H_qk = j1j2_hamiltonian(N, j1=J1, j2=J2)
spec = get_spectrum(N=N, j1=J1, j2=J2, source="dmrg")
E0, Etop = spec["E0"], spec["Etop"]
H_scaled_qk = ((H_qk + SparsePauliOp.from_list([("I" * N, -E0)]))
               / (Etop - E0)).simplify()
Hm = H_scaled_qk.to_matrix()

# ----------------------------------------------------------------------
# Optimize the filter (times, phases); T = sum of times
# ----------------------------------------------------------------------
builder = FilterBuilder(total_time=spec["totaltime"], energies=spec["energies"])
best = builder.build(method="v3")[-1]
sw_times, sw_phases = best["times"], best["phases"]
T = float(np.sum(sw_times))

# ----------------------------------------------------------------------
# Trial circuit
# ----------------------------------------------------------------------
trial_qc = mps_to_circuit(psi0_mps.arrays, method="approximate",
                          shape="lpr", num_layers=SWEEP_L)
if trial_qc.num_qubits != N:
    raise ValueError(f"trial circuit has {trial_qc.num_qubits} qubits, expected {N}")

print(f"\nFilter: {len(sw_times)} optimized pulses, T = sum t_i = {T:.4f} "
      f"(pi/gap = {spec['totaltime']:.4f}), t_i in "
      f"[{sw_times.min():.3f}, {sw_times.max():.3f}] (scaled-H units)")
print(f"\noriginal {len(sw_times)} pulse times (scaled-H units), sorted:")
for t in sorted(sw_times):
    print(f"  {t:.4f}")

# ----------------------------------------------------------------------
# Exact references
# ----------------------------------------------------------------------
p_ex, sv_ex = postselected_run_exact(trial_qc, sw_times, sw_phases, Hm)
E_ex, Fd_ex, Fe_ex = state_metrics(sv_ex, H_qk, psi0_dmrg, psi0_ed)
print(f"[exact, optimizer times]  P_succ={p_ex:.6f}  E-E0dmrg={E_ex - E0_dmrg:.3e}  "
      f"F_dmrg={Fd_ex:.6f}  F_ed={Fe_ex:.6f}")

# ----------------------------------------------------------------------
# Cost of one Trotter step (transpiled once), and the commutator constant
# ----------------------------------------------------------------------
cx1, d1 = cx_depth_per_step(H_scaled_qk, order=ORDER)
alpha = alpha_comm(H_scaled_qk, tight=True)
print(f"alpha (nested commutators) = {alpha:.4f}   CX/step = {cx1}   depth/step = {d1}")

# ----------------------------------------------------------------------
# Sweep over n: snap to grid, re-optimize phases, Trotterize with k_i steps
#
# bound_total = alpha * T^2 / (2n) is the standard first-order operator-norm
# Trotter bound (Childs et al. commutator scaling), evaluated with the tight
# grouped-commutator alpha, in the SAME term ordering LieTrotter uses.
#
# bound_state = 2 * bound_total / sqrt(p_g) is a HEURISTIC composition of that
# operator bound into a post-selected-STATE distance: for a single projection
# and renormalization step, ||a_hat - b_hat|| <= 2||a - b|| / ||a|| is a
# standard vector-normalization inequality, and here we treat the accumulated
# multi-pulse Trotter error as bounded by bound_total (by construction of
# alpha_comm/T) and p_g (the exact grid filter's success probability) as the
# reference norm ||a||. This is NOT a rigorously derived worst-case bound for
# sequential post-selection across many pulses -- it's a diagnostic check, and
# violations are reported as warnings, not treated as evidence of a bug.
# ----------------------------------------------------------------------
trot_rows = []
for n in N_SWEEP:
    k, dt, ph = grid_filter(sw_times, sw_phases, T, n, spec["energies"])
    tg = k * dt

    # exact filter on this grid: isolates the Trotter error from snapping loss
    p_g, sv_g = postselected_run_exact(trial_qc, tg, ph, Hm)
    _, Fd_g, Fe_g = state_metrics(sv_g, H_qk, psi0_dmrg, psi0_ed)

    p, sv = postselected_run(
        trial_qc, tg, ph,
        lambda qc, s, a, t, phi, i, k=k: apply_filter_pulse(
            qc, s, a, H_scaled_qk, t, phi, int(k[i]), order=ORDER))
    E, Fd, Fe = state_metrics(sv, H_qk, psi0_dmrg, psi0_ed)
    F_vs_grid = float(abs(np.vdot(sv_g, sv)) ** 2)

    bound_total = alpha * T ** 2 / (2 * n)
    bound_state = min(2.0, 2 * bound_total / np.sqrt(p_g))

    # stable state-error metric: direct phase-aligned vector norm (avoids the
    # precision floor of a fidelity-based sqrt(2(1-sqrt(F))) form near F ~ 1)
    theta = np.angle(np.vdot(sv_g, sv))
    state_err = float(np.linalg.norm(sv_g - np.exp(1j * theta) * sv))

    if state_err > bound_state + 1e-9:
        print(f"  [warning] n={n}: measured state error {state_err:.3e} "
              f"exceeds heuristic bound {bound_state:.3e} "
              f"(see bound derivation note above sweep loop)")

    row = {
        "n": n, "dt": dt, "n_pulses": len(k), "k": k.tolist(),
        "alpha": alpha, "bound_total": bound_total, "bound_state": bound_state,
        "state_err": state_err,
        "P_succ_exact_grid": p_g, "P_succ": p,
        "F_dmrg_exact_grid": Fd_g, "F_ed_exact_grid": Fe_g,
        "F_dmrg": Fd, "F_ed": Fe, "E-E0_dmrg": E - E0_dmrg,
        "F_vs_exact_grid": F_vs_grid,
        "cx": cx1 * n, "depth": d1 * n,          # estimates; validated below
    }
    trot_rows.append(row)

    print(f"[n={n:>4}] dt={dt:.4f} k={k.tolist()}  "
          f"t_i={np.round(tg, 4).tolist()}  "
          f"P_succ={p:.6f} (grid exact {p_g:.6f})  "
          f"F_dmrg={Fd:.6f} (grid exact {Fd_g:.6f})  "
          f"1-F_vs_grid={1 - F_vs_grid:.2e}  bound={bound_total:.2e}  "
          f"CX~{row['cx']}")

    if n in VERIFY_COST_NS:
        qc_r, _ = build_rodeo_circuit(H_scaled_qk, tg, ph,
                                      trial_prep=lambda qc, s: None,
                                      trotter_steps=[int(x) for x in k],
                                      order=ORDER)
        c = resource_costs(qc_r, label=f"n={n} (full transpile)", flatten=True)
        est_cx, meas_cx = row["cx"], c["cx"]
        pct = 100 * (meas_cx - est_cx) / est_cx if est_cx else float("nan")
        print(f"  cost check at n={n}: measured CX={meas_cx}, depth={c['depth']}  "
              f"vs estimate CX={est_cx}, depth={row['depth']}  "
              f"(estimate off by {pct:+.1f}%)")

# ----------------------------------------------------------------------
# Save
# ----------------------------------------------------------------------
with open("trotter_sweep.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(trot_rows[0].keys()))
    w.writeheader()
    w.writerows(trot_rows)

# ----------------------------------------------------------------------
# Plots
# ----------------------------------------------------------------------
ns = [r["n"] for r in trot_rows]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# 1) measured state error vs nested-commutator bound
ax[0].loglog(ns, [max(r["state_err"], 1e-16) for r in trot_rows], "o-",
             label="actual (phase-aligned vector norm)")
ax[0].loglog(ns, [r["bound_state"] for r in trot_rows], "k--",
             label=r"heuristic bound $2\alpha T^2/(2n\sqrt{p_g})$")
ax[0].set_xlabel("total Trotter steps n (dt = T/n)")
ax[0].set_ylabel("state error vs exact grid filter")
ax[0].set_title("Trotter error vs bound")
ax[0].legend()

# 2) fidelity and success probability
ax[1].semilogx(ns, [r["F_dmrg"] for r in trot_rows], "o-", color="C0",
               label="F vs DMRG (Trotter)")
ax[1].semilogx(ns, [r["F_dmrg_exact_grid"] for r in trot_rows], "o:", color="C0",
               alpha=0.6, label="F vs DMRG (exact, grid)")
ax[1].set_xlabel("total Trotter steps n")
ax[1].set_ylabel("post-filter fidelity", color="C0")
axr = ax[1].twinx()
axr.semilogx(ns, [r["P_succ"] for r in trot_rows], "s-", color="C1",
             label="P_succ (Trotter)")
axr.semilogx(ns, [r["P_succ_exact_grid"] for r in trot_rows], "s:", color="C1",
             alpha=0.6, label="P_succ (exact, grid)")
axr.set_ylabel("success probability", color="C1")
ax[1].set_title("Fidelity and success prob (dotted = exact)")
l1, n1 = ax[1].get_legend_handles_labels()
l2, n2 = axr.get_legend_handles_labels()
ax[1].legend(l1 + l2, n1 + n2, loc="lower right")

# 3) cost (estimate: per-step count x n)
ax[2].loglog(ns, [r["cx"] for r in trot_rows], "o-", label="CX (estimate)")
ax[2].loglog(ns, [r["depth"] for r in trot_rows], "s-", label="depth (estimate)")
ax[2].set_xlabel("total Trotter steps n")
ax[2].set_ylabel("count (basis h,s,cx,rz)")
ax[2].set_title("Filter cost (estimate; see cost-check prints for validation)")
ax[2].legend()

fig.suptitle(f"First-order Trotter, grid n sweep, N={N}, L={SWEEP_L}, J2/J1 = {J2 / J1}")
fig.tight_layout()
fig.savefig("trotter_sweep.png", dpi=150)
plt.close(fig)
print("saved trotter_sweep.csv, trotter_sweep.png")

/opt/conda/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/opt/conda/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


classical f0: [0.92060518 0.39049469]
quantum   f0: [0.92060518 0.39049469]
match: True
Running DMRG for ground state...
Running DMRG for highest excited state (via -H)...
  spectral bandwidth (E_top - E0) = 5.92474+1.33227e-15j, PENALTY = 6.51721+1.46549e-15j
Building orthogonal-projector penalty MPO from ground state...
Running DMRG for first excited state (PENALTY = 6.51721+1.46549e-15j)...
Running ED cross-check (N=8, dim=2^8=256)...
  [OK] E0: DMRG=-3.1247352417-0.0000000000j  ED=-3.1247352417  |diff|=4.93e-15
  [OK] E1: DMRG=-2.4825050111  ED=-2.4825050114  |diff|=2.88e-10
  [OK] E_top: DMRG=2.8000000000+0.0000000000j  ED=2.8000000000  |diff|=9.35e-15
E0 DMRG = -3.12473524   |<DMRG|ED>|^2 = 1.00000000
[source=dmrg] DMRG spectrum (E0, E1, E_top only), N=8
Running DMRG for ground state...
Running DMRG for highest excited state (via -H)...
  spectral bandwidth (E_top - E0) = 5.92474+3.33067e-16j, PENALTY = 6.51721+3.66374e-16j
Building orthogonal-projector penalty MPO from ground st